# Reinforcement Learning Agent Mathematical Models

> A rigorous treatment of the mathematical foundations that govern how RL agents learn, decide, and improve — from first principles to modern deep RL algorithms.

---

## Table of Contents

1. [The RL Framework: Markov Decision Process (MDP)](#1-the-rl-framework-markov-decision-process-mdp)
2. [The Agent's Core Objective: Return and Value](#2-the-agents-core-objective-return-and-value)
3. [Policy Models](#3-policy-models)
4. [Value Function Models](#4-value-function-models)
5. [The Bellman Equations](#5-the-bellman-equations)
6. [Model-Free Algorithms](#6-model-free-algorithms)
7. [Policy Gradient Methods](#7-policy-gradient-methods)
8. [Actor-Critic Architecture](#8-actor-critic-architecture)
9. [Proximal Policy Optimization (PPO)](#9-proximal-policy-optimization-ppo)
10. [Deep RL: Neural Network Function Approximation](#10-deep-rl-neural-network-function-approximation)
11. [Behavioral Dynamics: How Models Behave](#11-behavioral-dynamics-how-models-behave)
12. [Comparison of RL Agent Models](#12-comparison-of-rl-agent-models)

---

## 1. The RL Framework: Markov Decision Process (MDP)

Every reinforcement learning agent operates inside a **Markov Decision Process (MDP)**, the mathematical skeleton of sequential decision-making.

### 1.1 Formal Definition

An MDP is defined by the tuple:

```
M = (S, A, P, R, γ)
```

| Symbol | Name | Description |
|--------|------|-------------|
| `S` | State Space | The set of all possible environment states `s ∈ S` |
| `A` | Action Space | The set of all actions the agent can take `a ∈ A` |
| `P(s'\mid s, a)` | Transition Probability | Probability of landing in state `s'` after taking action `a` in state `s` |
| `R(s, a, s')` | Reward Function | Scalar signal received after each transition |
| `γ ∈ [0, 1)` | Discount Factor | How much future rewards are worth relative to immediate ones |

### 1.2 The Markov Property

The defining constraint of an MDP is that the **future is independent of the past, given the present**:

$$
P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, ...) = P(s_{t+1} \mid s_t, a_t)
$$

This means the current state `s_t` contains all information needed to predict the next state. The agent does not need memory of prior states.

### 1.3 The Agent-Environment Loop

```
         ┌─────────────────────────────────────────────┐
         │                                             │
  ┌──────▼──────┐   a_t = π(s_t)     ┌─────────────────┴──────┐
  │    AGENT    │ ─────────────────► │     ENVIRONMENT        │
  │  (policy π) │                    │                        │
  │             │ ◄───────────────── │  Transition: P(s'|s,a) │
  └─────────────┘   s_{t+1}, r_t     │  Reward: R(s, a, s')   │
                                     └────────────────────────┘
```

At each timestep `t`:
1. Agent observes state `s_t`
2. Agent selects action `a_t ~ π(·|s_t)`
3. Environment transitions to `s_{t+1} ~ P(·|s_t, a_t)`
4. Agent receives reward `r_t = R(s_t, a_t, s_{t+1})`

---

## 2. The Agent's Core Objective: Return and Value

### 2.1 Discounted Return

The agent's objective is to maximize the **expected discounted cumulative reward**, called the **return**:

$$
G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ... = Σ_{k=0}^{∞} γᵏ · r_{t+k}
$$

The discount factor `γ` controls temporal preference:

| `γ` value | Behavior |
|-----------|----------|
| `γ = 0` | Completely myopic — only cares about immediate reward |
| `γ → 1` | Far-sighted — future rewards weighted nearly equally |
| `γ = 0.99` | Typical value — balances short and long-term planning |

### 2.2 Why Discount?

Discounting serves three purposes:

- **Mathematical convergence** — ensures the infinite sum `G_t` is finite
- **Uncertainty** — future rewards are less certain than immediate ones
- **Economic interpretation** — mirrors time-preference of value (a reward now is worth more than a reward later)

---

## 3. Policy Models

The **policy** `π` is the agent's brain — it maps states to actions.

### 3.1 Deterministic Policy

$$
a = μ(s)
$$

A function that maps each state to a single, definite action. Used in algorithms like DDPG.

**Behavior:** The agent always takes the exact same action in the same state. Efficient but may get stuck in local optima.

### 3.2 Stochastic Policy

$$
a ~ π(a | s)     where  Σ_a π(a|s) = 1
$$

A **probability distribution** over actions given state `s`. The agent samples an action from this distribution.

**Discrete action spaces (Softmax policy):**

$$
π(a|s) = exp(h(s,a)) / Σ_{a'} exp(h(s,a'))
$$

where `h(s,a)` is a preference score (e.g., neural network logit).

**Continuous action spaces (Gaussian policy):**

$$
π(a|s) = N(a | μ_θ(s), σ_θ²(s))
$$

$$
         1               (a - μ_θ(s))²
= ─────────────── · exp(─ ────────────── )
  √(2π σ_θ²(s))            2σ_θ²(s)
$$

where `μ_θ(s)` is the mean action and `σ_θ(s)` is the standard deviation, both output by a neural network parameterized by `θ`.

**Behavior:** The agent explores naturally by sampling different actions. Entropy in the distribution controls the exploration-exploitation balance:

$$
H(π(·|s)) = -Σ_a π(a|s) · log π(a|s)     [discrete]

H(π(·|s)) = ½ · log(2πe · σ²(s))         [Gaussian]
$$

High entropy → broad exploration. Low entropy → confident exploitation.

---

## 4. Value Function Models

Value functions answer the question: **"How good is it to be in state `s` (or to take action `a` in state `s`)?"**

### 4.1 State-Value Function V(s)

$$
V^π(s) = E_π [ G_t | s_t = s ]
$$
$$
       = E_π [ Σ_{k=0}^{∞} γᵏ · r_{t+k} | s_t = s ]
$$

The expected return when starting from state `s` and following policy `π` forever after.

**Behavior:** `V^π(s)` is a scalar map over the entire state space. States the agent "likes" under policy `π` have high V; dangerous or dead-end states have low V.

### 4.2 Action-Value Function Q(s, a)

$$
Q^π(s, a) = E_π [ G_t | s_t = s, a_t = a ]
$$
$$
           = E_π [ Σ_{k=0}^{∞} γᵏ · r_{t+k} | s_t = s, a_t = a ]
$$

The expected return when starting from `s`, **forcing** the first action to be `a`, then following `π` thereafter.

**Relationship between V and Q:**

$$
V^π(s) = Σ_a π(a|s) · Q^π(s, a)     [discrete]

V^π(s) = E_{a ~ π(·|s)} [ Q^π(s, a) ]  [continuous]
$$

### 4.3 Advantage Function A(s, a)

$$
A^π(s, a) = Q^π(s, a) - V^π(s)
$$

The advantage measures how much **better (or worse)** action `a` is relative to the average action the policy would take in state `s`.

| Sign of A(s,a) | Meaning |
|----------------|---------|
| `A > 0` | Action `a` is better than average — increase its probability |
| `A = 0` | Action `a` is exactly average |
| `A < 0` | Action `a` is worse than average — decrease its probability |

**Behavior:** The advantage function is the signal that drives policy improvement. It separates "was this a good state?" from "was this a good action in that state?"

---

## 5. The Bellman Equations

The Bellman equations are **recursive decompositions** of value functions — the mathematical engine of most RL algorithms.

### 5.1 Bellman Expectation Equation (for V)

$$
V^π(s) = Σ_a π(a|s) · Σ_{s'} P(s'|s,a) · [ R(s,a,s') + γ · V^π(s') ]
          ──────────────────────────────────────────────────────────────
               Expected immediate reward + discounted future value
$$

In compact notation:

$$
V^π(s) = E_{a~π, s'~P} [ r + γ · V^π(s') ]
$$

### 5.2 Bellman Expectation Equation (for Q)

$$
Q^π(s,a) = Σ_{s'} P(s'|s,a) · [ R(s,a,s') + γ · Σ_{a'} π(a'|s') · Q^π(s',a') ]
$$

In compact notation:

$$
Q^π(s,a) = E_{s'~P} [ r + γ · E_{a'~π} [ Q^π(s',a') ] ]
$$

### 5.3 Bellman Optimality Equations

The **optimal** value functions `V*` and `Q*` satisfy:

$$
V*(s) = max_a Σ_{s'} P(s'|s,a) · [ R(s,a,s') + γ · V*(s') ]
$$
$$
Q*(s,a) = Σ_{s'} P(s'|s,a) · [ R(s,a,s') + γ · max_{a'} Q*(s',a') ]
$$

The optimal policy is then:

$$
π*(s) = argmax_a Q*(s,a)
$$

**Behavior:** These equations are the foundation of dynamic programming. The agent that satisfies the Bellman optimality equation is **provably optimal** — no other policy can achieve higher expected return.

---

## 6. Model-Free Algorithms

Model-free algorithms learn directly from experience without needing to know `P(s'|s,a)`.

### 6.1 Temporal Difference (TD) Learning

The core idea: **use current estimates to update current estimates** (bootstrapping).

**TD(0) Update Rule:**

$$
V(s_t) ← V(s_t) + α · [ r_t + γ·V(s_{t+1}) - V(s_t) ]
                         ─────────────────────────────
                              TD Error  δ_t
$$

The **TD error** `δ_t` is the key signal:

$$
δ_t = r_t + γ·V(s_{t+1}) - V(s_t)
$$

| `δ_t` sign | Meaning |
|------------|---------|
| `δ_t > 0` | The outcome was better than expected — increase V(s_t) |
| `δ_t = 0` | Perfect prediction — no update needed |
| `δ_t < 0` | The outcome was worse than expected — decrease V(s_t) |

**Behavior:** TD learning is sample-efficient because it updates after every step, not at the end of an episode. It converges to `V^π` for tabular environments under standard conditions.

### 6.2 Q-Learning

An **off-policy** algorithm that directly estimates `Q*`:

$$
Q(s_t, a_t) ← Q(s_t, a_t) + α · [ r_t + γ · max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) ]
                                    ──────────────────────────────────────────────────
                                              TD Target
$$

The critical difference from on-policy methods: the target uses `max_{a'}` regardless of what policy actually chose `a'`.

**Behavior:** Q-learning converges to `Q*` (the optimal Q-function) even when the agent explores with a random or ε-greedy policy. However, it overestimates Q-values due to the `max` operator.

### 6.3 SARSA (On-Policy TD)

$$
Q(s_t, a_t) ← Q(s_t, a_t) + α · [ r_t + γ · Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t) ]
$$

Unlike Q-learning, the update uses the actual next action `a_{t+1}` chosen by the policy.

**Behavior comparison:**

| | Q-Learning | SARSA |
|--|--|--|
| **Type** | Off-policy | On-policy |
| **Target** | `max Q(s', a')` | `Q(s', a')` where `a' ~ π` |
| **Learns** | Optimal Q* | Q under current policy |
| **Risk** | May be overconfident | More cautious near cliffs |

---

## 7. Policy Gradient Methods

Instead of learning a value function and deriving a policy, **policy gradient methods directly optimize the policy parameters `θ`**.

### 7.1 The Policy Gradient Theorem

The objective to maximize is the **expected return**:

$$
J(θ) = E_{τ ~ π_θ} [ G_0 ] = E_{τ ~ π_θ} [ Σ_t γᵗ · r_t ]
$$
$$
where `τ = (s_0, a_0, r_0, s_1, a_1, ...)` is a trajectory.
$$
The **policy gradient theorem** gives the gradient of `J(θ)`:

$$
∇_θ J(θ) = E_{τ ~ π_θ} [ Σ_t ∇_θ log π_θ(a_t|s_t) · G_t ]
$$

This is remarkable: we can compute the gradient **without knowing the transition probabilities `P`**.

**Derivation insight:**

$$
∇_θ π_θ(a|s) = π_θ(a|s) · ∇_θ log π_θ(a|s)
$$

This identity (the log-derivative trick) allows us to compute the gradient as an expectation.

### 7.2 REINFORCE Algorithm

The simplest policy gradient algorithm:

$$
For each episode:
$$
  1. Collect trajectory τ = {s_0, a_0, r_0, ..., s_T}
  2. For each timestep t:$$
       G_t = Σ_{k=t}^{T} γ^{k-t} · r_k$$
       θ ← θ + α · G_t · ∇_θ log π_θ(a_t|s_t)$$
$$

**Behavior:** REINFORCE is unbiased (the gradient estimate is correct in expectation) but has **very high variance** — returns `G_t` can differ dramatically between episodes even from the same state. This makes learning slow and unstable.

### 7.3 Baseline Subtraction (Variance Reduction)

$$
∇_θ J(θ) = E_τ [ Σ_t ∇_θ log π_θ(a_t|s_t) · (G_t - b(s_t)) ]
$$

Subtracting a **baseline** `b(s_t)` (often `V^π(s_t)`) does not bias the gradient (can be proven) but dramatically reduces variance.

When $$b(s_t) = V^π(s_t)`, the term `(G_t - V^π(s_t))$$ is an estimate of the **advantage** $$A^π(s_t, a_t)$$.

---

## 8. Actor-Critic Architecture

The actor-critic model combines value learning (critic) and policy optimization (actor) — getting the best of both worlds.

### 8.1 Architecture

```
         ┌─────────────────┐
State s  │   Shared Layers  │
────────►│   (CNN / MLP)   │
         └────────┬────────┘
                  │
       ┌──────────┴──────────┐
       ▼                     ▼
┌─────────────┐     ┌───────────────┐
│   ACTOR     │     │   CRITIC      │
│  π_θ(a|s)  │     │   V_φ(s)      │
│             │     │               │
│ Outputs:    │     │ Outputs:      │
│ action dist │     │ scalar value  │
└──────┬──────┘     └───────┬───────┘
       │                    │
       ▼                    ▼
    a_t ~ π          TD error δ_t
                       ↓
                   Update both θ and φ
```

### 8.2 The Update Equations

**Critic Update** (minimize TD error):

```
φ ← φ - α_c · ∇_φ [ r + γ·V_φ(s') - V_φ(s) ]²
```

**Actor Update** (ascend policy gradient):

```
θ ← θ + α_a · ∇_θ log π_θ(a|s) · A(s,a)
```

where the advantage estimate is:

```
Â(s_t, a_t) = r_t + γ·V_φ(s_{t+1}) - V_φ(s_t) = δ_t
```

### 8.3 Generalized Advantage Estimation (GAE)

A more sophisticated advantage estimator that trades off bias and variance:

```
Â_t^{GAE(γ,λ)} = Σ_{l=0}^{∞} (γλ)^l · δ_{t+l}
```

where `δ_{t+l} = r_{t+l} + γ·V(s_{t+l+1}) - V(s_{t+l})` is the TD error at step `t+l`.

| λ value | Effect |
|---------|--------|
| `λ = 0` | `Â_t = δ_t` (pure TD, low variance, high bias) |
| `λ = 1` | `Â_t = G_t - V(s_t)` (Monte Carlo, high variance, low bias) |
| `λ = 0.95` | Typical value — smooth bias-variance tradeoff |

**Behavior:** GAE allows tuning how many future steps contribute to the advantage estimate. Higher λ makes the agent more far-sighted but noisier in its gradient estimates.

---

## 9. Proximal Policy Optimization (PPO)

PPO is the algorithm used in the drone navigation system. It solves a critical problem: **how to update the policy aggressively without destabilizing training.**

### 9.1 The Core Problem PPO Solves

Large policy updates can collapse performance. The naive gradient step:

```
θ_{k+1} = θ_k + α · ∇_θ J(θ_k)
```

can move the policy so far that the new policy collects bad data, causing a catastrophic feedback loop.

### 9.2 The Probability Ratio

PPO defines the **importance sampling ratio**:

```
r_t(θ) = π_θ(a_t|s_t) / π_{θ_old}(a_t|s_t)
```

| `r_t(θ)` value | Meaning |
|----------------|---------|
| `r_t = 1` | Policy hasn't changed for this action |
| `r_t > 1` | New policy is MORE likely to take this action |
| `r_t < 1` | New policy is LESS likely to take this action |

The unclipped surrogate objective is:

```
L^{CPI}(θ) = E_t [ r_t(θ) · Â_t ]
```

### 9.3 The PPO-Clip Objective

PPO clips the ratio to prevent destructively large updates:

```
L^{CLIP}(θ) = E_t [ min( r_t(θ) · Â_t ,  clip(r_t(θ), 1-ε, 1+ε) · Â_t ) ]
```

The clipping behavior:

```
          ┌                                           ┐
          │  r_t · Â_t        if  1-ε ≤ r_t ≤ 1+ε   │
L^CLIP =  │  (1+ε) · Â_t     if  r_t > 1+ε (Â_t>0)  │  (capped)
          │  (1-ε) · Â_t     if  r_t < 1-ε (Â_t>0)  │  (floored)
          └                                           ┘
```

Visually, with `ε = 0.2`:

```
  L^CLIP
    ▲
    │           clipped region
    │     ┌──────────────────────────
    │     │
    │    /  unclipped region
    │   /
    │  /
    │ /
    └─────────────────────────────►  r_t(θ)
         1-ε  1.0  1+ε
```

### 9.4 Full PPO Objective

```
L^{PPO}(θ) = E_t [ L^{CLIP}(θ) - c₁·L^{VF}(θ) + c₂·H[π_θ(·|s_t)] ]
```

| Term | Role | Typical weight |
|------|------|----------------|
| `L^{CLIP}` | Policy gradient (clipped) | weight 1.0 |
| `L^{VF} = (V_θ(s_t) - V_t^{target})²` | Value function loss | `c₁ = 0.5` |
| `H[π_θ]` | Entropy bonus (encourages exploration) | `c₂ = 0.01` |

### 9.5 PPO Training Loop

```
Initialize policy π_θ and value function V_φ

Repeat:
  1. COLLECT: Run π_{θ_old} for T timesteps → {s_t, a_t, r_t, s_{t+1}}
  2. COMPUTE: GAE advantage estimates Â_t and value targets V_t^target
  3. OPTIMIZE: For K epochs, using minibatches of size M:
       θ ← θ + α · ∇_θ L^{PPO}(θ)
  4. UPDATE: θ_old ← θ

Until convergence
```

**Behavioral properties of PPO:**

- The clip constraint `ε` (typically 0.1–0.2) acts as a **trust region** — the policy cannot change too drastically in one update
- Multiple epochs `K` over the same data increase sample efficiency
- The entropy bonus `c₂·H` prevents the policy from collapsing to a single deterministic action prematurely

---

## 10. Deep RL: Neural Network Function Approximation

In practical environments like AirSim, the state space is enormous (pixel images, continuous vectors). We approximate value functions and policies with **neural networks**.

### 10.1 Function Approximation

Replace tabular functions with parameterized approximators:

```
V(s)        →   V_φ(s)        [neural network with weights φ]
Q(s,a)      →   Q_θ(s,a)      [neural network with weights θ]
π(a|s)      →   π_θ(a|s)      [neural network with weights θ]
```

### 10.2 Deep Q-Network (DQN)

The loss function for learning Q*:

```
L(θ) = E_{(s,a,r,s')~D} [ ( r + γ · max_{a'} Q_{θ⁻}(s',a') - Q_θ(s,a) )² ]
                                      ──────────────────────────────────
                                            TD Target (frozen weights θ⁻)
```

Key tricks that make deep Q-learning stable:

**Experience Replay Buffer `D`:**
```
D = {(s_1,a_1,r_1,s_1'), (s_2,a_2,r_2,s_2'), ..., (s_N,a_N,r_N,s_N')}
```
Sample random minibatches → breaks temporal correlations in data.

**Target Network `θ⁻`:**
```
θ⁻ ← θ        (updated only every C steps, not every gradient step)
```
Prevents chasing a moving target, stabilizes training.

### 10.3 Policy Network for Continuous Actions (PPO/SAC)

For continuous action spaces like drone velocity control `[vx, vy, vz]`:

```
Input: s_t = [pos, vel, waypoint_vector, depth_image]
         │
    ┌────▼─────────────┐
    │   CNN / MLP       │   (shared feature extractor)
    │   f_θ(s_t)        │
    └────┬─────────────┘
         │
    ┌────▼─────────────┐
    │   Policy Head     │
    │   μ_θ(s): R^n     │   → mean of Gaussian action
    │   σ_θ(s): R^n     │   → std deviation
    └───────────────────┘

   a_t ~ N( μ_θ(s_t), diag(σ_θ²(s_t)) )
```

For bounded action spaces `[-5, 5]^3` (drone velocities), a **tanh squash** is applied:

```
a_t = MAX_VEL · tanh(ã_t)       where  ã_t ~ N(μ_θ, σ_θ²)
```

with log-probability correction:

```
log π_θ(a_t|s_t) = log N(ã_t | μ_θ, σ_θ²) - Σ_i log(1 - tanh²(ã_{t,i}))
```

---

## 11. Behavioral Dynamics: How Models Behave

This section bridges the math to observable agent behavior.

### 11.1 Exploration vs. Exploitation

Every RL agent must balance two competing drives:

```
EXPLORATION                      EXPLOITATION
──────────────────               ──────────────────────
Try new, unknown actions         Use known-good actions
Gather information               Maximize immediate reward
High entropy policy              Low entropy (peaked) policy
```

**Mathematical measure of exploration entropy:**

```
H(π(·|s)) = -Σ_a π(a|s) · log π(a|s)
```

PPO uses entropy bonus `c₂·H` to maintain exploration throughout training.

**Behavioral trajectory:**

```
Training Progress ──────────────────────────────────────►

High Entropy              Medium Entropy          Low Entropy
(early training)          (mid training)          (late training)
   
π ≈ Uniform            π partially peaked        π ≈ Deterministic
Wide exploration         Balanced                 Focused exploitation
Low reward               Rising reward            High reward (converged)
```

### 11.2 The Reward Shaping Effect on Behavior

Each reward component in the drone environment sculpts a distinct behavioral tendency:

```
Reward Signal               Resulting Behavior
──────────────────────────  ──────────────────────────────────────
+C1·(prev_dist - curr_dist) Agent moves directly toward waypoints
                            (Greedy progress, straight-line paths)

-0.05 per step              Agent prefers short paths and speed
                            (Avoids unnecessary hovering)

-0.1 · ‖Δaction‖           Agent uses smooth velocity transitions
                            (No jerky or oscillatory flight)

+50 per waypoint            Agent treats each waypoint as a subgoal
                            (Hierarchical behavior emerges)

-100 collision              Agent learns to avoid building geometry
                            (Obstacle avoidance policy develops)

+100 / -100 landing         Agent modulates vz near the goal
                            (Precision landing behavior)
```

### 11.3 Convergence and Stability

**Convergence condition for TD methods (tabular):**

```
Σ_{t=0}^{∞} α_t = ∞        (learning rate doesn't vanish too fast)
Σ_{t=0}^{∞} α_t² < ∞       (but does vanish eventually)
```

A common schedule satisfying both: `α_t = α_0 / (1 + decay·t)`.

**Behavioral instability patterns:**

| Pattern | Mathematical Cause | Symptom |
|---------|-------------------|---------|
| Policy collapse | Entropy → 0 too fast | Agent repeats one action |
| Reward hacking | Misspecified reward | Unexpected shortcut behavior |
| Catastrophic forgetting | Non-stationary data distribution | Periodic performance crashes |
| Exploding gradients | `‖∇_θ L‖ → ∞` | NaN rewards, erratic flight |

### 11.4 The PPO Trust Region: Behavioral Constraint

The clip parameter `ε` directly controls **how different** consecutive policies can be. In behavior terms:

```
Small ε (0.05):   Conservative. Policy changes slowly. Safe but slow.
                  → Drone flight changes gradually between updates

Medium ε (0.2):   Standard. Good balance of speed and safety.
                  → Typical convergence in ~500k–2M steps

Large ε (0.5+):   Aggressive. Fast policy changes. Often unstable.
                  → May cause erratic flight or divergence
```

### 11.5 Hierarchical Behavior: Global Planner + Local RL

The two-level hierarchy in the drone system creates a behavioral decomposition:

```
GLOBAL LEVEL (A* Planner)
─────────────────────────
  - Operates in voxel space
  - Plans around known static obstacles (buildings)
  - Outputs: sequence of waypoints [W1, W2, ..., Wn]
  - Timescale: once per episode (or when replanning)
  - No learning involved — deterministic optimal planning

         ↓ waypoints

LOCAL LEVEL (PPO Agent)
────────────────────────
  - Operates in continuous world space
  - Tracks waypoints; avoids dynamic/unseen obstacles
  - Outputs: velocity commands [vx, vy, vz] at 10Hz
  - Timescale: every 0.1 seconds
  - Fully learned — adapts to sensor noise and disturbances
```

**Why hierarchy works:**

The A\* planner solves the **combinatorial navigation problem** globally but cannot react at 10Hz. The PPO agent solves the **local control problem** but cannot reason across 200×200×40 voxels. Together, they handle different **temporal and spatial scales** of the problem.

### 11.6 Value Function as an Agent's World Model

The value function `V^π(s)` can be visualized as a **landscape** over the state space:

```
 V(s)
  ▲
  │      Goal region (high value)
  │              ╭───╮
  │             /     \
  │            /       \
  │  Obstacles \       /  Safe corridors
  │   (low V)  \     /
  │          ─  \───/   ─
  │                      
  └─────────────────────► State space
```

The agent's behavior is determined by gradient ascent on `V^π(s)`: it naturally moves toward high-value regions (near waypoints, near goal) and away from low-value regions (near collisions, stagnant positions).

---

## 12. Comparison of RL Agent Models

| Property | Q-Learning | REINFORCE | A2C/A3C | PPO | SAC |
|----------|-----------|-----------|---------|-----|-----|
| **Policy type** | Implicit (greedy Q) | Stochastic | Stochastic | Stochastic | Stochastic |
| **On/Off policy** | Off | On | On | On | Off |
| **Action space** | Discrete | Both | Both | Both | Continuous |
| **Sample efficiency** | Medium | Low | Medium | Medium-High | High |
| **Stability** | Medium | Low | Medium | High | High |
| **Exploration** | ε-greedy | Natural stochasticity | Entropy bonus | Entropy bonus | Automatic (max-entropy) |
| **Parallelism** | No | No | Yes (A3C) | Yes | No |
| **Clip mechanism** | None | None | None | Trust region clip | KL regularization |
| **Best for** | Discrete tasks | Simple continuous | Multi-agent | Robotics / flight | Robotics / complex |

---

## Summary: The Mathematical Story of an RL Agent

```
1. DEFINE the task as an MDP (S, A, P, R, γ)
         ↓
2. REPRESENT the policy π_θ(a|s) as a neural network
         ↓
3. COLLECT experience by acting in the environment
         ↓
4. ESTIMATE value V_φ(s) and advantage Â(s,a) using Bellman equations
         ↓
5. COMPUTE policy gradient ∇_θ E[Â · log π_θ(a|s)]
         ↓
6. UPDATE policy with trust region constraint (PPO clip)
         ↓
7. REPEAT until convergence: J(θ) → J(π*) = max_π E[G_0]
```

The agent begins as a random wanderer in state space. Through millions of interactions, Bellman bootstrapping propagates reward signals backward through time, value estimates improve, and the policy gradient nudges action probabilities in directions that maximize return. The result is an autonomous agent that has — through pure mathematical optimization — learned to fly.

---

*Document covers: MDP foundations, policy and value function theory, Bellman equations, TD learning, Q-Learning, SARSA, policy gradients, actor-critic models, GAE, PPO, deep function approximation, and behavioral dynamics.*


# Reinforcement Learning Agent Mathematical Models

> A rigorous treatment of the mathematical foundations that govern how RL agents learn, decide, and improve — from first principles to modern deep RL algorithms.

---

## Table of Contents

1. [The RL Framework: Markov Decision Process (MDP)](#1-the-rl-framework-markov-decision-process-mdp)
2. [The Agent's Core Objective: Return and Value](#2-the-agents-core-objective-return-and-value)
3. [Policy Models](#3-policy-models)
4. [Value Function Models](#4-value-function-models)
5. [The Bellman Equations](#5-the-bellman-equations)
6. [Model-Free Algorithms](#6-model-free-algorithms)
7. [Policy Gradient Methods](#7-policy-gradient-methods)
8. [Actor-Critic Architecture](#8-actor-critic-architecture)
9. [Proximal Policy Optimization (PPO)](#9-proximal-policy-optimization-ppo)
10. [Deep RL: Neural Network Function Approximation](#10-deep-rl-neural-network-function-approximation)
11. [Behavioral Dynamics: How Models Behave](#11-behavioral-dynamics-how-models-behave)
12. [Comparison of RL Agent Models](#12-comparison-of-rl-agent-models)

---

## 1. The RL Framework: Markov Decision Process (MDP)

Every reinforcement learning agent operates inside a **Markov Decision Process (MDP)**, the mathematical skeleton of sequential decision-making.

### 1.1 Formal Definition

An MDP is defined by the tuple:

```
M = (S, A, P, R, γ)
```

| Symbol | Name | Description |
|--------|------|-------------|
| `S` | State Space | The set of all possible environment states `s ∈ S` |
| `A` | Action Space | The set of all actions the agent can take `a ∈ A` |
| `P(s'\mid s, a)` | Transition Probability | Probability of landing in state `s'` after taking action `a` in state `s` |
| `R(s, a, s')` | Reward Function | Scalar signal received after each transition |
| `γ ∈ [0, 1)` | Discount Factor | How much future rewards are worth relative to immediate ones |

### 1.2 The Markov Property

The defining constraint of an MDP is that the **future is independent of the past, given the present**:

$$
P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, ...) = P(s_{t+1} \mid s_t, a_t)
$$

This means the current state `s_t` contains all information needed to predict the next state. The agent does not need memory of prior states.

### 1.3 The Agent-Environment Loop

```
         ┌─────────────────────────────────────────────┐
         │                                             │
  ┌──────▼──────┐   a_t = π(s_t)     ┌─────────────────┴──────┐
  │    AGENT    │ ─────────────────► │     ENVIRONMENT        │
  │  (policy π) │                    │                        │
  │             │ ◄───────────────── │  Transition: P(s'|s,a) │
  └─────────────┘   s_{t+1}, r_t     │  Reward: R(s, a, s')   │
                                     └────────────────────────┘
```

At each timestep `t`:
1. Agent observes state `s_t`
2. Agent selects action `a_t ~ π(·|s_t)`
3. Environment transitions to `s_{t+1} ~ P(·|s_t, a_t)`
4. Agent receives reward `r_t = R(s_t, a_t, s_{t+1})`

---

## 2. The Agent's Core Objective: Return and Value

### 2.1 Discounted Return

The agent's objective is to maximize the **expected discounted cumulative reward**, called the **return**:

$$
G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ... = Σ_{k=0}^{∞} γᵏ · r_{t+k}
$$

The discount factor `γ` controls temporal preference:

| `γ` value | Behavior |
|-----------|----------|
| `γ = 0` | Completely myopic — only cares about immediate reward |
| `γ → 1` | Far-sighted — future rewards weighted nearly equally |
| `γ = 0.99` | Typical value — balances short and long-term planning |

### 2.2 Why Discount?

Discounting serves three purposes:

- **Mathematical convergence** — ensures the infinite sum `G_t` is finite
- **Uncertainty** — future rewards are less certain than immediate ones
- **Economic interpretation** — mirrors time-preference of value (a reward now is worth more than a reward later)

---

## 3. Policy Models

The **policy** `π` is the agent's brain — it maps states to actions.

### 3.1 Deterministic Policy

$$
a = μ(s)
$$

A function that maps each state to a single, definite action. Used in algorithms like DDPG.

**Behavior:** The agent always takes the exact same action in the same state. Efficient but may get stuck in local optima.

### 3.2 Stochastic Policy

$$
a ~ π(a | s)     where  Σ_a π(a|s) = 1
$$

A **probability distribution** over actions given state `s`. The agent samples an action from this distribution.

**Discrete action spaces (Softmax policy):**

$$
π(a|s) = exp(h(s,a)) / Σ_{a'} exp(h(s,a'))
$$

where `h(s,a)` is a preference score (e.g., neural network logit).

**Continuous action spaces (Gaussian policy):**

$$
π(a|s) = N(a | μ_θ(s), σ_θ²(s))
$$

$$
         1               (a - μ_θ(s))²
= ─────────────── · exp(─ ────────────── )
  √(2π σ_θ²(s))            2σ_θ²(s)
$$

where `μ_θ(s)` is the mean action and `σ_θ(s)` is the standard deviation, both output by a neural network parameterized by `θ`.

**Behavior:** The agent explores naturally by sampling different actions. Entropy in the distribution controls the exploration-exploitation balance:

$$
H(π(·|s)) = -Σ_a π(a|s) · log π(a|s)     [discrete]

H(π(·|s)) = ½ · log(2πe · σ²(s))         [Gaussian]
$$

High entropy → broad exploration. Low entropy → confident exploitation.

---

## 4. Value Function Models

Value functions answer the question: **"How good is it to be in state `s` (or to take action `a` in state `s`)?"**

### 4.1 State-Value Function V(s)

$$
V^π(s) = E_π [ G_t | s_t = s ]
$$
$$
       = E_π [ Σ_{k=0}^{∞} γᵏ · r_{t+k} | s_t = s ]
$$

The expected return when starting from state `s` and following policy `π` forever after.

**Behavior:** `V^π(s)` is a scalar map over the entire state space. States the agent "likes" under policy `π` have high V; dangerous or dead-end states have low V.

### 4.2 Action-Value Function Q(s, a)

$$
Q^π(s, a) = E_π [ G_t | s_t = s, a_t = a ]
$$
$$
           = E_π [ Σ_{k=0}^{∞} γᵏ · r_{t+k} | s_t = s, a_t = a ]
$$

The expected return when starting from `s`, **forcing** the first action to be `a`, then following `π` thereafter.

**Relationship between V and Q:**

$$
V^π(s) = Σ_a π(a|s) · Q^π(s, a)     [discrete]

V^π(s) = E_{a ~ π(·|s)} [ Q^π(s, a) ]  [continuous]
$$

### 4.3 Advantage Function A(s, a)

$$
A^π(s, a) = Q^π(s, a) - V^π(s)
$$

The advantage measures how much **better (or worse)** action `a` is relative to the average action the policy would take in state `s`.

| Sign of A(s,a) | Meaning |
|----------------|---------|
| `A > 0` | Action `a` is better than average — increase its probability |
| `A = 0` | Action `a` is exactly average |
| `A < 0` | Action `a` is worse than average — decrease its probability |

**Behavior:** The advantage function is the signal that drives policy improvement. It separates "was this a good state?" from "was this a good action in that state?"

---

## 5. The Bellman Equations

The Bellman equations are **recursive decompositions** of value functions — the mathematical engine of most RL algorithms.

### 5.1 Bellman Expectation Equation (for V)

$$
V^π(s) = Σ_a π(a|s) · Σ_{s'} P(s'|s,a) · [ R(s,a,s') + γ · V^π(s') ]
          ──────────────────────────────────────────────────────────────
               Expected immediate reward + discounted future value
$$

In compact notation:

$$
V^π(s) = E_{a~π, s'~P} [ r + γ · V^π(s') ]
$$

### 5.2 Bellman Expectation Equation (for Q)

$$
Q^π(s,a) = Σ_{s'} P(s'|s,a) · [ R(s,a,s') + γ · Σ_{a'} π(a'|s') · Q^π(s',a') ]
$$

In compact notation:

$$
Q^π(s,a) = E_{s'~P} [ r + γ · E_{a'~π} [ Q^π(s',a') ] ]
$$

### 5.3 Bellman Optimality Equations

The **optimal** value functions `V*` and `Q*` satisfy:

$$
V*(s) = max_a Σ_{s'} P(s'|s,a) · [ R(s,a,s') + γ · V*(s') ]
$$
$$
Q*(s,a) = Σ_{s'} P(s'|s,a) · [ R(s,a,s') + γ · max_{a'} Q*(s',a') ]
$$

The optimal policy is then:

$$
π*(s) = argmax_a Q*(s,a)
$$

**Behavior:** These equations are the foundation of dynamic programming. The agent that satisfies the Bellman optimality equation is **provably optimal** — no other policy can achieve higher expected return.

---

## 6. Model-Free Algorithms

Model-free algorithms learn directly from experience without needing to know `P(s'|s,a)`.

### 6.1 Temporal Difference (TD) Learning

The core idea: **use current estimates to update current estimates** (bootstrapping).

**TD(0) Update Rule:**

$$
V(s_t) ← V(s_t) + α · [ r_t + γ·V(s_{t+1}) - V(s_t) ]
                         ─────────────────────────────
                              TD Error  δ_t
$$

The **TD error** `δ_t` is the key signal:

$$
δ_t = r_t + γ·V(s_{t+1}) - V(s_t)
$$

| `δ_t` sign | Meaning |
|------------|---------|
| `δ_t > 0` | The outcome was better than expected — increase V(s_t) |
| `δ_t = 0` | Perfect prediction — no update needed |
| `δ_t < 0` | The outcome was worse than expected — decrease V(s_t) |

**Behavior:** TD learning is sample-efficient because it updates after every step, not at the end of an episode. It converges to `V^π` for tabular environments under standard conditions.

### 6.2 Q-Learning

An **off-policy** algorithm that directly estimates `Q*`:

$$
Q(s_t, a_t) ← Q(s_t, a_t) + α · [ r_t + γ · max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) ]
                                    ──────────────────────────────────────────────────
                                              TD Target
$$

The critical difference from on-policy methods: the target uses `max_{a'}` regardless of what policy actually chose `a'`.

**Behavior:** Q-learning converges to `Q*` (the optimal Q-function) even when the agent explores with a random or ε-greedy policy. However, it overestimates Q-values due to the `max` operator.

### 6.3 SARSA (On-Policy TD)

$$
Q(s_t, a_t) ← Q(s_t, a_t) + α · [ r_t + γ · Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t) ]
$$

Unlike Q-learning, the update uses the actual next action `a_{t+1}` chosen by the policy.

**Behavior comparison:**

| | Q-Learning | SARSA |
|--|--|--|
| **Type** | Off-policy | On-policy |
| **Target** | `max Q(s', a')` | `Q(s', a')` where `a' ~ π` |
| **Learns** | Optimal Q* | Q under current policy |
| **Risk** | May be overconfident | More cautious near cliffs |

---

## 7. Policy Gradient Methods

Instead of learning a value function and deriving a policy, **policy gradient methods directly optimize the policy parameters `θ`**.

### 7.1 The Policy Gradient Theorem

The objective to maximize is the **expected return**:

$$
J(θ) = E_{τ ~ π_θ} [ G_0 ] = E_{τ ~ π_θ} [ Σ_t γᵗ · r_t ]
$$
$$
where `τ = (s_0, a_0, r_0, s_1, a_1, ...)` is a trajectory.
$$
The **policy gradient theorem** gives the gradient of `J(θ)`:

$$
∇_θ J(θ) = E_{τ ~ π_θ} [ Σ_t ∇_θ log π_θ(a_t|s_t) · G_t ]
$$

This is remarkable: we can compute the gradient **without knowing the transition probabilities `P`**.

**Derivation insight:**

$$
∇_θ π_θ(a|s) = π_θ(a|s) · ∇_θ log π_θ(a|s)
$$

This identity (the log-derivative trick) allows us to compute the gradient as an expectation.

### 7.2 REINFORCE Algorithm

The simplest policy gradient algorithm:

$$
For each episode:
$$
  1. Collect trajectory τ = {s_0, a_0, r_0, ..., s_T}
  2. For each timestep t:$$
       G_t = Σ_{k=t}^{T} γ^{k-t} · r_k$$
       θ ← θ + α · G_t · ∇_θ log π_θ(a_t|s_t)$$
$$

**Behavior:** REINFORCE is unbiased (the gradient estimate is correct in expectation) but has **very high variance** — returns `G_t` can differ dramatically between episodes even from the same state. This makes learning slow and unstable.

### 7.3 Baseline Subtraction (Variance Reduction)

$$
∇_θ J(θ) = E_τ [ Σ_t ∇_θ log π_θ(a_t|s_t) · (G_t - b(s_t)) ]
$$

Subtracting a **baseline** `b(s_t)` (often `V^π(s_t)`) does not bias the gradient (can be proven) but dramatically reduces variance.

When $$b(s_t) = V^π(s_t)`, the term `(G_t - V^π(s_t))$$ is an estimate of the **advantage** $$A^π(s_t, a_t)$$.

---

## 8. Actor-Critic Architecture

The actor-critic model combines value learning (critic) and policy optimization (actor) — getting the best of both worlds.

### 8.1 Architecture

```
         ┌─────────────────┐
State s  │   Shared Layers  │
────────►│   (CNN / MLP)   │
         └────────┬────────┘
                  │
       ┌──────────┴──────────┐
       ▼                     ▼
┌─────────────┐     ┌───────────────┐
│   ACTOR     │     │   CRITIC      │
│  π_θ(a|s)  │     │   V_φ(s)      │
│             │     │               │
│ Outputs:    │     │ Outputs:      │
│ action dist │     │ scalar value  │
└──────┬──────┘     └───────┬───────┘
       │                    │
       ▼                    ▼
    a_t ~ π          TD error δ_t
                       ↓
                   Update both θ and φ
```

### 8.2 The Update Equations

**Critic Update** (minimize TD error):

```
φ ← φ - α_c · ∇_φ [ r + γ·V_φ(s') - V_φ(s) ]²
```

**Actor Update** (ascend policy gradient):

```
θ ← θ + α_a · ∇_θ log π_θ(a|s) · A(s,a)
```

where the advantage estimate is:

```
Â(s_t, a_t) = r_t + γ·V_φ(s_{t+1}) - V_φ(s_t) = δ_t
```

### 8.3 Generalized Advantage Estimation (GAE)

A more sophisticated advantage estimator that trades off bias and variance:

```
Â_t^{GAE(γ,λ)} = Σ_{l=0}^{∞} (γλ)^l · δ_{t+l}
```

where `δ_{t+l} = r_{t+l} + γ·V(s_{t+l+1}) - V(s_{t+l})` is the TD error at step `t+l`.

| λ value | Effect |
|---------|--------|
| `λ = 0` | `Â_t = δ_t` (pure TD, low variance, high bias) |
| `λ = 1` | `Â_t = G_t - V(s_t)` (Monte Carlo, high variance, low bias) |
| `λ = 0.95` | Typical value — smooth bias-variance tradeoff |

**Behavior:** GAE allows tuning how many future steps contribute to the advantage estimate. Higher λ makes the agent more far-sighted but noisier in its gradient estimates.

---

## 9. Proximal Policy Optimization (PPO)

PPO is the algorithm used in the drone navigation system. It solves a critical problem: **how to update the policy aggressively without destabilizing training.**

### 9.1 The Core Problem PPO Solves

Large policy updates can collapse performance. The naive gradient step:

```
θ_{k+1} = θ_k + α · ∇_θ J(θ_k)
```

can move the policy so far that the new policy collects bad data, causing a catastrophic feedback loop.

### 9.2 The Probability Ratio

PPO defines the **importance sampling ratio**:

```
r_t(θ) = π_θ(a_t|s_t) / π_{θ_old}(a_t|s_t)
```

| `r_t(θ)` value | Meaning |
|----------------|---------|
| `r_t = 1` | Policy hasn't changed for this action |
| `r_t > 1` | New policy is MORE likely to take this action |
| `r_t < 1` | New policy is LESS likely to take this action |

The unclipped surrogate objective is:

```
L^{CPI}(θ) = E_t [ r_t(θ) · Â_t ]
```

### 9.3 The PPO-Clip Objective

PPO clips the ratio to prevent destructively large updates:

```
L^{CLIP}(θ) = E_t [ min( r_t(θ) · Â_t ,  clip(r_t(θ), 1-ε, 1+ε) · Â_t ) ]
```

The clipping behavior:

```
          ┌                                           ┐
          │  r_t · Â_t        if  1-ε ≤ r_t ≤ 1+ε   │
L^CLIP =  │  (1+ε) · Â_t     if  r_t > 1+ε (Â_t>0)  │  (capped)
          │  (1-ε) · Â_t     if  r_t < 1-ε (Â_t>0)  │  (floored)
          └                                           ┘
```

Visually, with `ε = 0.2`:

```
  L^CLIP
    ▲
    │           clipped region
    │     ┌──────────────────────────
    │     │
    │    /  unclipped region
    │   /
    │  /
    │ /
    └─────────────────────────────►  r_t(θ)
         1-ε  1.0  1+ε
```

### 9.4 Full PPO Objective

```
L^{PPO}(θ) = E_t [ L^{CLIP}(θ) - c₁·L^{VF}(θ) + c₂·H[π_θ(·|s_t)] ]
```

| Term | Role | Typical weight |
|------|------|----------------|
| `L^{CLIP}` | Policy gradient (clipped) | weight 1.0 |
| `L^{VF} = (V_θ(s_t) - V_t^{target})²` | Value function loss | `c₁ = 0.5` |
| `H[π_θ]` | Entropy bonus (encourages exploration) | `c₂ = 0.01` |

### 9.5 PPO Training Loop

```
Initialize policy π_θ and value function V_φ

Repeat:
  1. COLLECT: Run π_{θ_old} for T timesteps → {s_t, a_t, r_t, s_{t+1}}
  2. COMPUTE: GAE advantage estimates Â_t and value targets V_t^target
  3. OPTIMIZE: For K epochs, using minibatches of size M:
       θ ← θ + α · ∇_θ L^{PPO}(θ)
  4. UPDATE: θ_old ← θ

Until convergence
```

**Behavioral properties of PPO:**

- The clip constraint `ε` (typically 0.1–0.2) acts as a **trust region** — the policy cannot change too drastically in one update
- Multiple epochs `K` over the same data increase sample efficiency
- The entropy bonus `c₂·H` prevents the policy from collapsing to a single deterministic action prematurely

---

## 10. Deep RL: Neural Network Function Approximation

In practical environments like AirSim, the state space is enormous (pixel images, continuous vectors). We approximate value functions and policies with **neural networks**.

### 10.1 Function Approximation

Replace tabular functions with parameterized approximators:

```
V(s)        →   V_φ(s)        [neural network with weights φ]
Q(s,a)      →   Q_θ(s,a)      [neural network with weights θ]
π(a|s)      →   π_θ(a|s)      [neural network with weights θ]
```

### 10.2 Deep Q-Network (DQN)

The loss function for learning Q*:

```
L(θ) = E_{(s,a,r,s')~D} [ ( r + γ · max_{a'} Q_{θ⁻}(s',a') - Q_θ(s,a) )² ]
                                      ──────────────────────────────────
                                            TD Target (frozen weights θ⁻)
```

Key tricks that make deep Q-learning stable:

**Experience Replay Buffer `D`:**
```
D = {(s_1,a_1,r_1,s_1'), (s_2,a_2,r_2,s_2'), ..., (s_N,a_N,r_N,s_N')}
```
Sample random minibatches → breaks temporal correlations in data.

**Target Network `θ⁻`:**
```
θ⁻ ← θ        (updated only every C steps, not every gradient step)
```
Prevents chasing a moving target, stabilizes training.

### 10.3 Policy Network for Continuous Actions (PPO/SAC)

For continuous action spaces like drone velocity control `[vx, vy, vz]`:

```
Input: s_t = [pos, vel, waypoint_vector, depth_image]
         │
    ┌────▼─────────────┐
    │   CNN / MLP       │   (shared feature extractor)
    │   f_θ(s_t)        │
    └────┬─────────────┘
         │
    ┌────▼─────────────┐
    │   Policy Head     │
    │   μ_θ(s): R^n     │   → mean of Gaussian action
    │   σ_θ(s): R^n     │   → std deviation
    └───────────────────┘

   a_t ~ N( μ_θ(s_t), diag(σ_θ²(s_t)) )
```

For bounded action spaces `[-5, 5]^3` (drone velocities), a **tanh squash** is applied:

```
a_t = MAX_VEL · tanh(ã_t)       where  ã_t ~ N(μ_θ, σ_θ²)
```

with log-probability correction:

```
log π_θ(a_t|s_t) = log N(ã_t | μ_θ, σ_θ²) - Σ_i log(1 - tanh²(ã_{t,i}))
```

---

## 11. Behavioral Dynamics: How Models Behave

This section bridges the math to observable agent behavior.

### 11.1 Exploration vs. Exploitation

Every RL agent must balance two competing drives:

```
EXPLORATION                      EXPLOITATION
──────────────────               ──────────────────────
Try new, unknown actions         Use known-good actions
Gather information               Maximize immediate reward
High entropy policy              Low entropy (peaked) policy
```

**Mathematical measure of exploration entropy:**

```
H(π(·|s)) = -Σ_a π(a|s) · log π(a|s)
```

PPO uses entropy bonus `c₂·H` to maintain exploration throughout training.

**Behavioral trajectory:**

```
Training Progress ──────────────────────────────────────►

High Entropy              Medium Entropy          Low Entropy
(early training)          (mid training)          (late training)
   
π ≈ Uniform            π partially peaked        π ≈ Deterministic
Wide exploration         Balanced                 Focused exploitation
Low reward               Rising reward            High reward (converged)
```

### 11.2 The Reward Shaping Effect on Behavior

Each reward component in the drone environment sculpts a distinct behavioral tendency:

```
Reward Signal               Resulting Behavior
──────────────────────────  ──────────────────────────────────────
+C1·(prev_dist - curr_dist) Agent moves directly toward waypoints
                            (Greedy progress, straight-line paths)

-0.05 per step              Agent prefers short paths and speed
                            (Avoids unnecessary hovering)

-0.1 · ‖Δaction‖           Agent uses smooth velocity transitions
                            (No jerky or oscillatory flight)

+50 per waypoint            Agent treats each waypoint as a subgoal
                            (Hierarchical behavior emerges)

-100 collision              Agent learns to avoid building geometry
                            (Obstacle avoidance policy develops)

+100 / -100 landing         Agent modulates vz near the goal
                            (Precision landing behavior)
```

### 11.3 Convergence and Stability

**Convergence condition for TD methods (tabular):**

```
Σ_{t=0}^{∞} α_t = ∞        (learning rate doesn't vanish too fast)
Σ_{t=0}^{∞} α_t² < ∞       (but does vanish eventually)
```

A common schedule satisfying both: `α_t = α_0 / (1 + decay·t)`.

**Behavioral instability patterns:**

| Pattern | Mathematical Cause | Symptom |
|---------|-------------------|---------|
| Policy collapse | Entropy → 0 too fast | Agent repeats one action |
| Reward hacking | Misspecified reward | Unexpected shortcut behavior |
| Catastrophic forgetting | Non-stationary data distribution | Periodic performance crashes |
| Exploding gradients | `‖∇_θ L‖ → ∞` | NaN rewards, erratic flight |

### 11.4 The PPO Trust Region: Behavioral Constraint

The clip parameter `ε` directly controls **how different** consecutive policies can be. In behavior terms:

```
Small ε (0.05):   Conservative. Policy changes slowly. Safe but slow.
                  → Drone flight changes gradually between updates

Medium ε (0.2):   Standard. Good balance of speed and safety.
                  → Typical convergence in ~500k–2M steps

Large ε (0.5+):   Aggressive. Fast policy changes. Often unstable.
                  → May cause erratic flight or divergence
```

### 11.5 Hierarchical Behavior: Global Planner + Local RL

The two-level hierarchy in the drone system creates a behavioral decomposition:

```
GLOBAL LEVEL (A* Planner)
─────────────────────────
  - Operates in voxel space
  - Plans around known static obstacles (buildings)
  - Outputs: sequence of waypoints [W1, W2, ..., Wn]
  - Timescale: once per episode (or when replanning)
  - No learning involved — deterministic optimal planning

         ↓ waypoints

LOCAL LEVEL (PPO Agent)
────────────────────────
  - Operates in continuous world space
  - Tracks waypoints; avoids dynamic/unseen obstacles
  - Outputs: velocity commands [vx, vy, vz] at 10Hz
  - Timescale: every 0.1 seconds
  - Fully learned — adapts to sensor noise and disturbances
```

**Why hierarchy works:**

The A\* planner solves the **combinatorial navigation problem** globally but cannot react at 10Hz. The PPO agent solves the **local control problem** but cannot reason across 200×200×40 voxels. Together, they handle different **temporal and spatial scales** of the problem.

### 11.6 Value Function as an Agent's World Model

The value function `V^π(s)` can be visualized as a **landscape** over the state space:

```
 V(s)
  ▲
  │      Goal region (high value)
  │              ╭───╮
  │             /     \
  │            /       \
  │  Obstacles \       /  Safe corridors
  │   (low V)  \     /
  │          ─  \───/   ─
  │                      
  └─────────────────────► State space
```

The agent's behavior is determined by gradient ascent on `V^π(s)`: it naturally moves toward high-value regions (near waypoints, near goal) and away from low-value regions (near collisions, stagnant positions).

---

## 12. Comparison of RL Agent Models

| Property | Q-Learning | REINFORCE | A2C/A3C | PPO | SAC |
|----------|-----------|-----------|---------|-----|-----|
| **Policy type** | Implicit (greedy Q) | Stochastic | Stochastic | Stochastic | Stochastic |
| **On/Off policy** | Off | On | On | On | Off |
| **Action space** | Discrete | Both | Both | Both | Continuous |
| **Sample efficiency** | Medium | Low | Medium | Medium-High | High |
| **Stability** | Medium | Low | Medium | High | High |
| **Exploration** | ε-greedy | Natural stochasticity | Entropy bonus | Entropy bonus | Automatic (max-entropy) |
| **Parallelism** | No | No | Yes (A3C) | Yes | No |
| **Clip mechanism** | None | None | None | Trust region clip | KL regularization |
| **Best for** | Discrete tasks | Simple continuous | Multi-agent | Robotics / flight | Robotics / complex |

---

## Summary: The Mathematical Story of an RL Agent

```
1. DEFINE the task as an MDP (S, A, P, R, γ)
         ↓
2. REPRESENT the policy π_θ(a|s) as a neural network
         ↓
3. COLLECT experience by acting in the environment
         ↓
4. ESTIMATE value V_φ(s) and advantage Â(s,a) using Bellman equations
         ↓
5. COMPUTE policy gradient ∇_θ E[Â · log π_θ(a|s)]
         ↓
6. UPDATE policy with trust region constraint (PPO clip)
         ↓
7. REPEAT until convergence: J(θ) → J(π*) = max_π E[G_0]
```

The agent begins as a random wanderer in state space. Through millions of interactions, Bellman bootstrapping propagates reward signals backward through time, value estimates improve, and the policy gradient nudges action probabilities in directions that maximize return. The result is an autonomous agent that has — through pure mathematical optimization — learned to fly.

---

*Document covers: MDP foundations, policy and value function theory, Bellman equations, TD learning, Q-Learning, SARSA, policy gradients, actor-critic models, GAE, PPO, deep function approximation, and behavioral dynamics.*
